### Try an analysis

In [ ]:
import os
os.environ['PISA_RESOURCES'] = "../"
os.environ['PISA_RESOURCES'] += os.pathsep + "../../"
os.environ['PISA_RESOURCES'] += os.pathsep + "/data/ana/LE/"

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import math
import pickle
import numpy as np
from uncertainties import unumpy as unp
import matplotlib.pyplot as plt
import pisa
import copy
from scipy.interpolate import RectBivariateSpline
from scipy.optimize import minimize
from scipy import stats

from pisa.core.pipeline import Pipeline
from pisa.core.distribution_maker import DistributionMaker
from pisa.core.detectors import Detectors
from pisa import FTYPE, ureg
from pisa.utils.fileio import from_file
from pisa.core.map import MapSet
from pisa.analysis.analysis import update_param_values, update_param_values_detector

In [ ]:
params = {'legend.fontsize': 18,
          'figure.figsize': (7, 7*0.618),
          'axes.labelsize': 18,
          'axes.titlesize': 18,
          'xtick.labelsize': 18,
          'ytick.labelsize': 18}
plt.rcParams.update(params)

In [ ]:
def plot_map(templates, cmap='Spectral_r', **kwargs):        

    if templates.binning.num_dims == 2:
        templates.plot(cmap=cmap, **kwargs);
        return
    
    n_pid = templates.binning.pid.num_bins
    
    fig, ax = plt.subplots(1, n_pid, figsize=((n_pid+1)*5,5))
    plt.subplots_adjust(wspace=0.3)

    for pid in range(n_pid):
        templates.slice(pid=pid).squeeze().plot(ax = ax[pid], cmap=cmap, **kwargs);
        ax[pid].set_title(f'PID {pid}')

def area(C):
    vs = C.collections[0].get_paths()[0].vertices
    x, y = vs[:,0], vs[:,1]
    return 0.5*np.abs(np.dot(x,np.roll(y,1))-np.dot(y,np.roll(x,1)))

In [ ]:
%%time
#template_maker = DistributionMaker(["settings/pipeline/pipeline_upgrade_neutrinos_std_osc_NO.cfg", "settings/pipeline/pipeline_upgrade_muons.cfg"], profile=True)
#template_maker = DistributionMaker(["settings/pipeline/pipeline_oscnext_neutrinos_std_osc_NO.cfg", "settings/pipeline/pipeline_oscnext_muons.cfg"], profile=True)


p1_nu = Pipeline("settings/pipeline/pipeline_upgrade_neutrinos_std_osc_NO.cfg")
p1_mu = Pipeline("settings/pipeline/pipeline_upgrade_muons.cfg")
p2_nu = Pipeline("settings/pipeline/pipeline_oscnext_neutrinos_std_osc_NO.cfg")
#p2_mu = Pipeline("settings/pipeline/pipeline_oscnext_muons.cfg")

shared_params = list(p1_nu.params.free.names) #+ list(p1_mu.params.free.names)
shared_params.remove('icu_dom_eff')
template_maker = Detectors([p1_nu, p1_mu, p2_nu], shared_params=shared_params, profile=True)

In [ ]:
%%time
template_maker.params.reset_free()
templates = template_maker.get_outputs()

In [ ]:
if template_maker.__class__.__name__ == "Detectors":
    print(template_maker.distribution_makers[0].pipelines[0].report_profile())
else:
    print(template_maker.pipelines[0].report_profile())

In [ ]:
template_maker.params.free

### Maps

In [ ]:
if template_maker.__class__.__name__ == "Detectors":
    for i in range(len(templates)):
        combined = templates[i][0].combine_wildcard('*')
        name = template_maker.distribution_makers[i].detector_name
        print(f"Total number of "+name+f" events in hist: {combined.hist.sum().nominal_value}")
        plot_map(combined)
        plt.gcf().suptitle(name+' (%.1f events)'%(combined.hist.sum().nominal_value), y=0.98, fontsize=20)
        #plt.savefig('../plots/maps_'+name, bbox_inches='tight')
else:
    combined = templates[0].combine_wildcard('*')
    print(f"Total number of events in hist: {combined.hist.sum().nominal_value}")
    plot_map(combined)
    plt.gcf().suptitle('(%.1f events)'%(combined.hist.sum().nominal_value), y=0.98, fontsize=20)
    #plt.savefig('../plots/maps', bbox_inches='tight')

In [ ]:
if template_maker.__class__.__name__ == "Detectors":
    for i in range(len(templates)):
        name = template_maker.distribution_makers[i].detector_name
        fig, axes = plt.subplots(3,4, figsize=(34,15))
        plt.subplots_adjust(hspace=0.5)
        axes = axes.flatten()
        plt.gcf().suptitle(name, y=0.98, fontsize=20)

        for m, ax in zip(templates[i][0], axes):
            m.plot(ax=ax)
        #plt.savefig('../plots/maps_'+name+'_all', bbox_inches='tight')
else:
    fig, axes = plt.subplots(3,4, figsize=(34,15))
    plt.subplots_adjust(hspace=0.5)
    axes = axes.flatten()

    for m, ax in zip(templates[0], axes):
        m.plot(ax=ax)
    #plt.savefig('../plots/maps_all', bbox_inches='tight')

### Fake data

In [ ]:
fake_data = template_maker.get_outputs(return_sum=True)

In [ ]:
if template_maker.__class__.__name__ == "Detectors":
    for i in range(len(templates)):
        fake_data[i].plot()
else:
    fake_data.plot()

### Try a quick $\chi^2$ scan without changing systematics

In [ ]:
def get_mod_chi2(data, template_maker, theta23=None, deltam31=None):
    params = template_maker.params
    
    if theta23 is not None:
        params['theta23'].value = theta23 * ureg.degrees
    
    if deltam31 is not None:
        params['deltam31'].value = deltam31 * ureg.electron_volt**2
        
    if template_maker.__class__.__name__ == "Detectors":
        update_param_values_detector(template_maker, params)
        metrics, template = [], template_maker.get_outputs()
        for i in range(len(template)):
            metrics.append(data[i][0].mod_chi2(template[i]))
        return np.sum(metrics) + template_maker.params.priors_penalty('mod_chi2'), metrics
    else:
        update_param_values(template_maker, params)
        return data[0].mod_chi2(template_maker.get_outputs()) + template_maker.params.priors_penalty('mod_chi2')

In [ ]:
%%time
print(get_mod_chi2(fake_data, template_maker, 45.6, 0.0023))

template_maker.params.reset_free()
if template_maker.__class__.__name__ == "Detectors":
    update_param_values_detector(template_maker, template_maker.params)

In [ ]:
def theta_from_sinsq(sin_sq):
    return math.asin(math.sqrt(sin_sq)) * 180 / math.pi

def scan_chi2(data, template_maker, gridsize=11):
    sin_sqs = np.linspace(0.4, 0.6, gridsize)
    delta_ms = np.linspace(2.35e-3, 2.6e-3, gridsize)

    chi2_scan = np.zeros((gridsize, gridsize))

    if template_maker.__class__.__name__ == "Detectors":
        chi2_scan_det1 = np.zeros((gridsize, gridsize))
        chi2_scan_det2 = np.zeros((gridsize, gridsize))
        for i, sin_sq in enumerate(sin_sqs):
            for j, delta_m in enumerate(delta_ms):
                chi2s = get_mod_chi2(data, template_maker, theta_from_sinsq(sin_sq), delta_m)
                chi2_scan[j, i] = chi2s[0]
                chi2_scan_det1[j, i] = chi2s[1][0]
                chi2_scan_det2[j, i] = chi2s[1][1]
        return chi2_scan, chi2_scan_det1, chi2_scan_det2, sin_sqs, delta_ms
    else:
        for i, sin_sq in enumerate(sin_sqs):
            for j, delta_m in enumerate(delta_ms):
                chi2_scan[j, i] = get_mod_chi2(data, template_maker, theta_from_sinsq(sin_sq), delta_m)      
        return chi2_scan, sin_sqs, delta_ms

In [ ]:
%%time
if template_maker.__class__.__name__ == "Detectors":
    chi2, chi2_det1, chi2_det2, sin_sqs, delta_ms = scan_chi2(fake_data, template_maker, gridsize=11)
else:
    chi2, sin_sqs, delta_ms = scan_chi2(fake_data, template_maker, gridsize=11)

template_maker.params.reset_free()
if template_maker.__class__.__name__ == "Detectors":
    update_param_values_detector(template_maker, template_maker.params)

In [ ]:
#save scan
if template_maker.__class__.__name__ == "Detectors":
    d = {'chi2': chi2, 'chi2_det1': chi2_det1, 'chi2_det2': chi2_det2, 'sin_sqs': sin_sqs, 'delta_ms': delta_ms}
else:
    d = {'chi2': chi2, 'sin_sqs': sin_sqs, 'delta_ms': delta_ms}

with open("../scans/quick_scan_dynedge.pkl", "wb") as f:
    pickle.dump(d, f)

#### plot contours

In [ ]:
with open("../scans/quick_scan_dynedge.pkl", "rb") as f:
    d = pickle.load(f)

sin_sqs = d['sin_sqs']
delta_ms = d['delta_ms']

if template_maker.__class__.__name__ == "Detectors":
    chi2 = d['chi2']
    chi2_det1 = d['chi2_det1']
    chi2_det2 = d['chi2_det2']
else:
    chi2 = d['chi2']
    
with open("../scans/quick_scan_DC.pkl", "rb") as f:
    d = pickle.load(f)
chi2_DC = d['chi2']

In [ ]:
interp = RectBivariateSpline(sin_sqs, delta_ms, chi2)

interp_xs = np.linspace(sin_sqs[0], sin_sqs[-1], 100)
interp_ys = np.linspace(delta_ms[0], delta_ms[-1], 100)
interp_zs = interp(interp_xs, interp_ys)

if template_maker.__class__.__name__ == "Detectors":
    interp_det1 = RectBivariateSpline(sin_sqs, delta_ms, chi2_det1)
    interp_det2 = RectBivariateSpline(sin_sqs, delta_ms, chi2_det2)
    
    interp_zs_det1 = interp_det1(interp_xs, interp_ys)
    interp_zs_det2 = interp_det2(interp_xs, interp_ys)
    
interp_DC = RectBivariateSpline(sin_sqs, delta_ms, chi2_DC)
interp_zs_DC = interp_DC(interp_xs, interp_ys)

In [ ]:
level, lab = 4.61, '90%'
fig = plt.figure(figsize=(9, 9*0.618))

if template_maker.__class__.__name__ == "Detectors":
    C = plt.contour(interp_xs, interp_ys, interp_zs_det2, [level], linestyles=['--'], colors=['Tab:orange'])
    plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
    plt.plot([], [], label='DeepCore (12yr)', linestyle='--', color='Tab:orange')
    
    C = plt.contour(interp_xs, interp_ys, interp_zs_det1, [level], linestyles=['--'], colors=['Tab:blue'])
    plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
    plt.plot([], [], label='DeepCore + new strings (3yr)', linestyle='--', color='Tab:blue')
    
C = plt.contour(interp_xs, interp_ys, interp_zs, [level], colors=['black'], linewidths=[2])
plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
plt.plot([], [], label='2028 with new strings', color='black', linewidth=2)

C = plt.contour(interp_xs, interp_ys, interp_zs_DC, [level], colors=['black'], linewidths=[2], linestyles=[':'])
plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
plt.plot([], [], label='2028 without new strings', color='black', linewidth=2, linestyle=':')

plt.scatter(np.sin(template_maker.params.theta23.nominal_value.magnitude * math.pi/180)**2, 
            template_maker.params.deltam31.nominal_value.magnitude, 
            color='r', marker='x', label='Injected truth')

plt.title('Fixed systematics')
plt.legend(prop={'size':13}, loc='lower right') #, bbox_to_anchor = [1.05, 1.05]
plt.xlabel(r'$\sin^2(\theta_{23})$')
plt.ylabel('$\Delta m_{31}^2$ ($e\mathrm{V}^2$)', fontsize=16)
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

#plt.savefig('../plots/quick_scan_dynedge.png', bbox_inches='tight')

In [ ]:
def scan_chi2(data, template_maker, gridsize=11, mode=0):
    delta_ms = np.linspace(-2.45e-3, -2.3e-3, gridsize)
    sin_sqs = np.linspace(0.4, 0.6, gridsize)
    chi2_scan = np.zeros(gridsize)

    chi2_scan_det1 = np.zeros(gridsize)
    chi2_scan_det2 = np.zeros(gridsize)
    if mode == 0:
        for j, delta_m in enumerate(delta_ms):
            chi2s = get_mod_chi2(data, template_maker, deltam31=delta_m)
            chi2_scan[j] = chi2s[0]
            chi2_scan_det1[j] = chi2s[1][0]
            chi2_scan_det2[j] = chi2s[1][1]
    elif mode == 1:
        for j, sin_sq in enumerate(sin_sqs):
            chi2s = get_mod_chi2(data, template_maker, theta23=theta_from_sinsq(sin_sq))
            chi2_scan[j] = chi2s[0]
            chi2_scan_det1[j] = chi2s[1][0]
            chi2_scan_det2[j] = chi2s[1][1]
            
    template_maker.params.reset_free()
    update_param_values_detector(template_maker, template_maker.params)
    
    return chi2_scan, chi2_scan_det1, chi2_scan_det2, delta_ms, sin_sqs

In [ ]:
template_maker.select_params('ih')
template_maker.params.reset_free()

In [ ]:
chi2, chi2_det1, chi2_det2, delta_ms, sin_sqs = scan_chi2(fake_data, template_maker, gridsize=30)
chi2_t, chi2_det1_t, chi2_det2_t, delta_ms, sin_sqs = scan_chi2(fake_data, template_maker, gridsize=30, mode=1)

In [ ]:
plt.figure(figsize=(14, 7*0.618))

plt.subplot(121)
plt.plot(delta_ms, chi2_det1, label='Upgrade')
plt.plot(delta_ms, chi2_det2, label='DeepCore')

plt.axvline(delta_ms[np.argmin(chi2_det1)], color='Tab:blue')
plt.axvline(delta_ms[np.argmin(chi2_det2)], color='Tab:orange')

plt.legend()
plt.title('NO=True, IO for scan, fixed sys')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.xlabel('$\Delta m_{31}^2$ ($e\mathrm{V}^2$)')
plt.ylabel('$\chi^{2}_{mod}$')
plt.ylim(0,25)

plt.subplot(122)
plt.plot(sin_sqs, chi2_det1_t, label='Upgrade')
plt.plot(sin_sqs, chi2_det2_t, label='DeepCore')

plt.axvline(sin_sqs[np.argmin(chi2_det1_t)], color='Tab:blue')
plt.axvline(sin_sqs[np.argmin(chi2_det2_t)], color='Tab:orange')

plt.legend()
plt.title('NO=True, IO for scan, fixed sys')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.xlabel(r'$\sin^2(\theta_{23})$')
plt.ylabel('$\chi^{2}_{mod}$')
#plt.ylim(0,25)

#plt.savefig('../plots/NMO_synergy.png', bbox_inches='tight')